# Quick Model Diagnostic

A lightweight version of the testing framework to quickly check feasibility and diagnostics.

In [1]:
import numpy as np
import pandas as pd
import time
import cvxpy as cp

from cma.data_reader import (
    read_vessel_class_data,
    read_port_data,
    read_sailing_distance_data,
    read_demand_with_transit_time,
    read_cnc_proforma_data,
)
from cma.port import PortGraph
from cma.servicegraph import ServiceGraph

np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## 1. Load Data (Minimal Subset)

In [2]:
vesselpool = read_vessel_class_data()
portpool_main, portpool_dmd = read_port_data()
dist_matrix = read_sailing_distance_data(portpool_main)
demand_matrix, transit_time_matrix = read_demand_with_transit_time(portpool_main)

portgraph = PortGraph(
    portpool_main, 
    dist_matrix, 
    demand_matrix,
    mat_transit_time=transit_time_matrix,
    filter_by_demand=False
)

proforma = read_cnc_proforma_data(portpool_main, vesselpool)
all_service_lines = proforma['lines']

# SUBSET FOR SPEED
service_lines = all_service_lines[:10]
servicegraph = ServiceGraph(service_lines)

trans_ports = portgraph.filtered_by_transship_capacity()
od_pairs_dict = servicegraph.get_all_paths(portgraph, trans_ports)

all_od_pairs = od_pairs_dict['od_pairs']
all_demands = od_pairs_dict['od_pairs_demand']
all_paths = od_pairs_dict['od_pairs_path']

filtered_pairs = []
filtered_paths = []
for od, paths, dmd in zip(all_od_pairs, all_paths, all_demands):
    if dmd > 0:
        filtered_pairs.append(od)
        filtered_paths.append(paths)

# SUBSET OD PAIRS
od_pairs = filtered_pairs[:50]
od_paths = filtered_paths[:50]

print(f"Loaded {len(service_lines)} lines and {len(od_pairs)} OD pairs for quick test.")

Loaded 10 lines and 50 OD pairs for quick test.


## 2. Run Optimization (All Features)

In [3]:
tuneparams = {
    'turnon-transship_shipclass_restriction': 1,
    'turnon-vessel_speed_optimization': 0,       # 0=ON (accurate)
    'turnon-port_operations_constraint': 1,
    'turnon-transit_time_penalty': 1,
    'ctrparam-kts_buffer': 0,
    'ctrparam-transship_A': 100,
    'ctrparam-speed_soft_cap_kts': 16.5,
    'ctrparam-speed_penalty_multiplier': 2.0,
    'ctrparam-transit_penalty_multiplier': 1000.0,
    'ctrparam-buffer_penalty_below_15pct': 1000.0,
    'ctrparam-buffer_penalty_above_30pct': 2000.0,
    'BigM-transship': 10000,
    'BigM-n_ships': 10,
    'BigM-saildays': 100,
    'BigM-line_capacity': 3000,
    'BigM-portcall_cost': 1e7,          
    'solver-MIPGap': 0.05,              # 5% gap (returns "good enough" results much faster)
    'solver-TimeLimit': 21600,          
    'solver-MIPFocus': 1,               # Focus on finding feasible solutions quickly
    'solver-verbose': True              # Show Gurobi's progress logs
}

week_levels = [1, 2, 3, 4, 5, 6, 7, 8]

In [4]:
start_time = time.time()
solution = servicegraph.fulfill_demands(
    od_pairs,
    od_paths,
    portgraph,
    vesselpool,
    week_levels,
    tuneparams
)
end_time = time.time()

print(f"Solve Time: {end_time - start_time:.2f}s")
print(f"Total Cost: {solution['total cost']}")

c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 1 times so far.

  warnings.warn(msg, UserWarning)
c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 2 times so far.

  warnings.warn(msg, UserWarning)
c:\Use

                                     CVXPY                                     
                                     v1.7.5                                    


(CVXPY) Feb 13 06:53:40 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Feb 13 06:53:40 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Feb 13 06:53:40 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Feb 13 06:53:40 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Feb 13 06:53:40 PM: Compiling problem (target solver=GUROBI).
(CVXPY) Feb 13 06:53:40 PM: Reduction chain: CvxAttr2Constr -> Qp2SymbolicQp -> QpMatrixStuffing -> GUROBI
(CVXPY) Feb 13 06:53:40 PM: Applying reduction CvxAttr2Constr


-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Feb 13 06:53:41 PM: Applying reduction Qp2SymbolicQp
(CVXPY) Feb 13 06:53:42 PM: Applying reduction QpMatrixStuffing
(CVXPY) Feb 13 06:53:46 PM: Applying reduction GUROBI
(CVXPY) Feb 13 06:53:46 PM: Finished problem compilation (took 6.691e+00 seconds).
(CVXPY) Feb 13 06:53:46 PM: Invoking solver GUROBI  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
Set parameter Username
Set parameter LicenseID to value 2725074
Academic license - for non-commercial use only - expires 2026-10-20
Set parameter OutputFlag to value 1
Set parameter QCPDual to value 1
Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 21600
Set parameter MIPFocus to value 1
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11.0 (26100.2))

CPU model: AMD Ryzen 7 5700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  21600
MIPGap  0.05
MIPFocus  1
QCPDual  1

Optimize a model with 15042 rows, 6891 columns and 116088 nonzeros
Model fingerprint: 0x256dde07
Variable types: 4701 continuous

(CVXPY) Feb 13 06:55:10 PM: Problem status: optimal


-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------


(CVXPY) Feb 13 06:55:10 PM: Optimal value: 1.073e+09
(CVXPY) Feb 13 06:55:10 PM: Compilation took 6.691e+00 seconds
(CVXPY) Feb 13 06:55:10 PM: Solver (including time spent in interface) took 8.323e+01 seconds


Solve Time: 92.15s
Total Cost: 1073140616.522352


## 3. Diagnostics & Soft Constraint Analysis

In [5]:
if solution['total cost'] == float('inf'):
    print("❌ STILL INFEASIBLE. Checking hard constraints...")
    # ... (similar check as in test_buffer_constraint.ipynb) ...
else:
    print("✓ FEASIBLE! Analyzing soft constraint violations:")
    
    # Analyze Buffer Violations
    violation_lb = solution['buffer violation lb']
    violation_ub = solution['buffer violation ub']
    
    violation_data = []
    for i, line in enumerate(service_lines):
        lb_val = violation_lb[i].value if hasattr(violation_lb[i], 'value') else 0
        ub_val = violation_ub[i].value if hasattr(violation_ub[i], 'value') else 0
        if lb_val > 0.01 or ub_val > 0.01:
            violation_data.append({
                'Line': line.name(),
                'Below 15% (h)': lb_val,
                'Above 30% (h)': ub_val
            })
    
    if violation_data:
        print("\nBuffer Violations Detected:")
        print(pd.DataFrame(violation_data))
    else:
        print("\nNo buffer violations! All lines within 15-30% range.")

    # Analyze Weeks/Vessels picked
    weeks_vars = solution['weeks']
    picked_weeks = []
    for i, line in enumerate(service_lines):
        for j, wk in enumerate(week_levels):
            if weeks_vars[i, j].value > 0.5:
                picked_weeks.append({'Line': line.name(), 'Week': wk})
    
    print("\nCycle Times (Weeks) Picked:")
    print(pd.DataFrame(picked_weeks))

✓ FEASIBLE! Analyzing soft constraint violations:

Buffer Violations Detected:
      Line  Below 15% (h)  Above 30% (h)
0  BBX3CNC       0.000000      84.663661
1   BBXCNC       0.000000       2.321186
2   BMXCNC       0.000000      65.697957
3  CHN1CNC       0.000000      51.336779
4   CP3CNC       0.000000       2.359413
5   CP8CNC       0.000000      14.267312
6   CS1CNC      26.036841       0.000000

Cycle Times (Weeks) Picked:
      Line  Week
0  BBX2CNC     3
1  BBX3CNC     3
2   BBXCNC     2
3   BMXCNC     4
4  CHN1CNC     3
5  CMS2CNC     2
6   CP2CNC     2
7   CP3CNC     2
8   CP8CNC     1
9   CS1CNC     2


## 4. Full Dataset Evaluation

Testing the model on all service lines and all positive demand OD pairs to identify potential bottlenecks.

In [6]:
service_lines_full = all_service_lines
servicegraph_full = ServiceGraph(service_lines_full)

print("Finding all paths for full dataset...")
trans_ports = portgraph.filtered_by_transship_capacity()
od_pairs_dict_full = servicegraph_full.get_all_paths(portgraph, trans_ports)

all_od_pairs_full = od_pairs_dict_full['od_pairs']
all_demands_full = od_pairs_dict_full['od_pairs_demand']
all_paths_full = od_pairs_dict_full['od_pairs_path']

filtered_pairs_full = []
filtered_paths_full = []
for od, paths, dmd in zip(all_od_pairs_full, all_paths_full, all_demands_full):
    if dmd > 0:
        filtered_pairs_full.append(od)
        filtered_paths_full.append(paths)

print(f"Full Dataset: {len(service_lines_full)} lines and {len(filtered_pairs_full)} OD pairs.")

Finding all paths for full dataset...
Full Dataset: 31 lines and 741 OD pairs.


In [7]:
start_time = time.time()
solution_full = servicegraph_full.fulfill_demands(
    filtered_pairs_full,
    filtered_paths_full,
    portgraph,
    vesselpool,
    week_levels,
    tuneparams
)
end_time = time.time()

print(f"Full Solve Time: {end_time - start_time:.2f}s")
print(f"Total Cost: {solution_full['total cost']}")

c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 11 times so far.

  warnings.warn(msg, UserWarning)
c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 12 times so far.

  warnings.warn(msg, UserWarning)
c:\U

                                     CVXPY                                     
                                     v1.7.5                                    


(CVXPY) Feb 13 06:55:20 PM: Your problem has 31924 variables, 47075 constraints, and 0 parameters.
(CVXPY) Feb 13 06:55:21 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Feb 13 06:55:21 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Feb 13 06:55:21 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Feb 13 06:55:21 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Feb 13 06:55:24 PM: Compiling problem (target solver=GUROBI).
(CVXPY) Feb 13 06:55:24 PM: Reduction chain: CvxAttr2Constr -> Qp2SymbolicQp -> QpMatrixStuffing -> GUROBI
(CVXPY) Feb 13 06:55:24 PM: Applying reduction CvxAttr2Constr


-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Feb 13 06:55:28 PM: Applying reduction Qp2SymbolicQp
(CVXPY) Feb 13 06:55:34 PM: Applying reduction QpMatrixStuffing
(CVXPY) Feb 13 07:01:05 PM: Applying reduction GUROBI
(CVXPY) Feb 13 07:01:05 PM: Finished problem compilation (took 3.435e+02 seconds).
(CVXPY) Feb 13 07:01:05 PM: Invoking solver GUROBI  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
Set parameter OutputFlag to value 1
Set parameter QCPDual to value 1
Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 21600
Set parameter MIPFocus to value 1
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11.0 (26100.2))

CPU model: AMD Ryzen 7 5700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  21600
MIPGap  0.05
MIPFocus  1
QCPDual  1

Optimize a model with 47075 rows, 31924 columns and 545555 nonzeros
Model fingerprint: 0xc63ff932
Variable types: 25135 continuous, 6789 integer (6448 binary)
Coefficient statistics:
  Matrix range     [4e-05, 1e+07]
  Objective range  [1e-01, 1e+06]
  Bounds 

(CVXPY) Feb 13 07:06:56 PM: Problem status: optimal
(CVXPY) Feb 13 07:06:56 PM: Optimal value: 8.804e+09
(CVXPY) Feb 13 07:06:56 PM: Compilation took 3.435e+02 seconds
(CVXPY) Feb 13 07:06:56 PM: Solver (including time spent in interface) took 3.506e+02 seconds


-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
Full Solve Time: 706.61s
Total Cost: 8803757052.95125


In [8]:
if solution_full['total cost'] == float('inf'):
    print("\n❌ FULL DATASET INFEASIBLE. Identifying problematic components...")
    
    # 1. Check for ports with zero productivity that have demand
    zero_prod_ports = []
    for port in portgraph.tolist_port():
        prods = port.get_producticity(vesselpool)
        if sum(prods) == 0:
            zero_prod_ports.append(port.get_id())
    
    if zero_prod_ports:
        print(f"\nPorts with ZERO productivity: {zero_prod_ports}")
        # Check if any OD pair involves these ports
        for i, (o_idx, d_idx) in enumerate(filtered_pairs_full):
            o_id = portgraph.get_port_by_idx(o_idx).get_id()
            d_id = portgraph.get_port_by_idx(d_idx).get_id()
            if o_id in zero_prod_ports or d_id in zero_prod_ports:
                print(f"  ⚠️ OD Pair {o_id}->{d_id} involves zero-productivity port.")

    # 2. Check for impossible distance/speed requirements (Hard Constraints)
    print("\nChecking for distance/speed violations:")
    for line in service_lines_full:
        dist = line.get_distance(portgraph)
        max_weeks = max(week_levels)
        min_speed_needed = dist / (24 * (7 * max_weeks - 1.0)) # 1 day min stay
        if min_speed_needed > 18.5:
            print(f"  ❌ Line {line.name()}: {dist:.0f}nm needs {min_speed_needed:.1f} kts at {max_weeks} weeks (Max 18.5)")

else:
    print("\n✓ FULL DATASET FEASIBLE!")
    
    # Violation Summary
    violation_lb = solution_full['buffer violation lb']
    violation_ub = solution_full['buffer violation ub']
    violation_summary = []
    for i, line in enumerate(service_lines_full):
        lb = violation_lb[i].value if hasattr(violation_lb[i], 'value') else 0
        ub = violation_ub[i].value if hasattr(violation_ub[i], 'value') else 0
        if lb > 0.1 or ub > 0.1:
            violation_summary.append({'Line': line.name(), 'LB_Violation': lb, 'UB_Violation': ub})
    
    if violation_summary:
        print("\nSignificant Buffer Violations in Full Dataset:")
        print(pd.DataFrame(violation_summary))


✓ FULL DATASET FEASIBLE!

Significant Buffer Violations in Full Dataset:
       Line  LB_Violation  UB_Violation
0   BBX3CNC      0.000000     84.663661
1    BBXCNC      0.000000      2.321186
2    BMXCNC      0.000000     97.537996
3    CP2CNC      0.000000      6.347184
4    CP3CNC      0.000000      2.359413
5    CP8CNC      0.000000      4.042614
6    CS1CNC     26.036841      0.000000
7    CV8CNC      0.000000      4.472209
8    KCSCNC      8.600000      0.000000
9   SGS2CNC      0.000000      4.199653
10   SGSCNC      0.000000      0.199653
11   YCXCNC      0.000000     14.057139
12   YSXCNC      0.000000      7.200149


## 5. Speed-Simplified Model (Full Dataset)

Running the model where vessel speed optimization is simplified (turnon-vessel_speed_optimization > 1/2).

In [9]:
tuneparams_simple = tuneparams.copy()
tuneparams_simple['turnon-vessel_speed_optimization'] = 1  # 1 = OFF (simplified)

start_time = time.time()
solution_simple = servicegraph_full.fulfill_demands(
    filtered_pairs_full,
    filtered_paths_full,
    portgraph,
    vesselpool,
    week_levels,
    tuneparams_simple
)
end_time = time.time()

print(f"Simple Model Solve Time: {end_time - start_time:.2f}s")
print(f"Total Cost: {solution_simple['total cost']}")

c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 42 times so far.

  warnings.warn(msg, UserWarning)
c:\Users\ASUS\.conda\envs\py311\Lib\site-packages\cvxpy\expressions\expression.py:683: UserWarning: 
This use of ``*`` has resulted in matrix multiplication.
Using ``*`` for matrix multiplication has been deprecated since CVXPY 1.1.
    Use ``*`` for matrix-scalar and vector-scalar multiplication.
    Use ``@`` for matrix-matrix and matrix-vector multiplication.
    Use ``multiply`` for elementwise multiplication.
This code path has been hit 43 times so far.

  warnings.warn(msg, UserWarning)
c:\U

                                     CVXPY                                     
                                     v1.7.5                                    


(CVXPY) Feb 13 07:07:06 PM: Your problem has 24688 variables, 20332 constraints, and 0 parameters.
(CVXPY) Feb 13 07:07:07 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Feb 13 07:07:07 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Feb 13 07:07:07 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Feb 13 07:07:07 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Feb 13 07:07:09 PM: Compiling problem (target solver=GUROBI).
(CVXPY) Feb 13 07:07:09 PM: Reduction chain: CvxAttr2Constr -> Qp2SymbolicQp -> QpMatrixStuffing -> GUROBI
(CVXPY) Feb 13 07:07:09 PM: Applying reduction CvxAttr2Constr


-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Feb 13 07:07:12 PM: Applying reduction Qp2SymbolicQp
(CVXPY) Feb 13 07:07:18 PM: Applying reduction QpMatrixStuffing
(CVXPY) Feb 13 07:10:19 PM: Applying reduction GUROBI
(CVXPY) Feb 13 07:10:19 PM: Finished problem compilation (took 1.918e+02 seconds).
(CVXPY) Feb 13 07:10:19 PM: Invoking solver GUROBI  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
Set parameter OutputFlag to value 1
Set parameter QCPDual to value 1
Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 21600
Set parameter MIPFocus to value 1
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11.0 (26100.2))

CPU model: AMD Ryzen 7 5700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  21600
MIPGap  0.05
MIPFocus  1
QCPDual  1

Optimize a model with 20332 rows, 24688 columns and 292945 nonzeros
Model fingerprint: 0x491a769f
Variable types: 18457 continuous, 6231 integer (5890 binary)
Coefficient statistics:
  Matrix range     [4e-05, 1e+07]
  Objective range  [1e-01, 1e+06]
  Bounds 

(CVXPY) Feb 13 07:10:32 PM: Problem status: optimal
(CVXPY) Feb 13 07:10:32 PM: Optimal value: 8.812e+09
(CVXPY) Feb 13 07:10:32 PM: Compilation took 1.918e+02 seconds
(CVXPY) Feb 13 07:10:32 PM: Solver (including time spent in interface) took 1.240e+01 seconds


-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
Simple Model Solve Time: 215.79s
Total Cost: 8811986941.79443


In [10]:
if solution_simple['total cost'] == float('inf'):
    print("\n❌ SIMPLE MODEL INFEASIBLE.")
else:
    print("\n✓ SIMPLE MODEL FEASIBLE!")
    
    # Violation Summary
    violation_lb = solution_simple['buffer violation lb']
    violation_ub = solution_simple['buffer violation ub']
    violation_summary = []
    for i, line in enumerate(service_lines_full):
        lb = violation_lb[i].value if hasattr(violation_lb[i], 'value') else 0
        ub = violation_ub[i].value if hasattr(violation_ub[i], 'value') else 0
        if lb > 0.1 or ub > 0.1:
            violation_summary.append({'Line': line.name(), 'LB_Violation': lb, 'UB_Violation': ub})
    
    if violation_summary:
        print("\nSignificant Buffer Violations in Simple Model:")
        print(pd.DataFrame(violation_summary))


✓ SIMPLE MODEL FEASIBLE!

Significant Buffer Violations in Simple Model:
     Line  LB_Violation  UB_Violation
0  CS1CNC     22.066542           0.0
1  KCSCNC      2.904118           0.0
